# Protocolo Experimental: Cinemática de Vaciado de un Tanque (Ley de Torricelli)

Este notebook documenta y ejecuta la caracterización del vaciado de un tanque cilíndrico utilizando un sensor ultrasónico HC-SR04 y una placa Arduino.

## Medidas del sistema físico
Antes de comenzar, registre las siguientes dimensiones:
* Diámetro del tanque: 21.0 cm
* Diámetro del orificio de salida: 6.0 mm
* Altura inicial de la columna de agua (h0): 6.0 cm
* Distancia inicial fija del sensor al agua: 9.5 cm
* Distancia fija del sensor al orificio (dH): 15.5 cm

In [ ]:
import serial
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import os

# --- 1. CONFIGURACIÓN DEL EXPERIMENTO ---
PUERTO = 'COM4'          # Puerto en Windows
BAUD_RATE = 9600
TIEMPO_VACIADO = 160.0   # Cálculo hidrodinámico: ~135s para vaciar 6cm en un tanque de 21cm
SIGMA_US = 0.3           # Incertidumbre del ultrasónico (cm)
UMBRAL_GATILLO = 0.2     # Diferencia en cm (2 mm) para disparar el inicio

# --- 2. EL MODELO FÍSICO (Parábola de Torricelli) ---
def modelo_torricelli(t, A, B, C):
    return A * t**2 + B * t + C

# --- 3. CAPTURA DE DATOS VÍA SERIAL (CON GATILLO AUTOMÁTICO) ---
tiempos = []
distancias = []
d_inicial_promedio = 9.5 # Valor por defecto por si falla la captura en vivo

print("Conectando al microcontrolador...")
try:
    with serial.Serial(PUERTO, BAUD_RATE, timeout=1) as ser:
        print("\n[FASE 1] Calculando línea base de distancia (NO quite el tapón)...")
        baseline_reads = []
        
        while len(baseline_reads) < 20:
            if ser.in_waiting > 0:
                linea = ser.readline().decode('utf-8').strip()
                try:
                    t_val, d_val = map(float, linea.split(','))
                    if 8.0 < d_val < 16.5:
                        baseline_reads.append(d_val)
                except ValueError:
                    pass
                    
        if len(baseline_reads) > 0:
            d_inicial_promedio = np.mean(baseline_reads)
        print(f"Línea base fijada en: {d_inicial_promedio:.2f} cm")
        
        input("\nPresione ENTER y quite el tapón en cualquier momento...")
        print("Esperando caída de agua para disparar cronómetro...")
        
        triggered = False
        start_time = 0
        
        while True:
            if ser.in_waiting > 0:
                linea = ser.readline().decode('utf-8').strip()
                try:
                    t_val, d_val = map(float, linea.split(','))
                    
                    if 8.0 < d_val < 16.5:
                        if not triggered:
                            if d_val > (d_inicial_promedio + UMBRAL_GATILLO):
                                triggered = True
                                start_time = time.time()
                                print(f"¡GATILLO ACTIVADO! Grabando {TIEMPO_VACIADO} segundos...")
                                tiempos.append(0.0)
                                distancias.append(d_val)
                        else:
                            t_actual = time.time() - start_time
                            if t_actual <= TIEMPO_VACIADO:
                                tiempos.append(t_actual)
                                distancias.append(d_val)
                            else:
                                print("\n¡Tiempo de grabación finalizado!")
                                break 
                except ValueError:
                    pass
except Exception as e:
    print(f"Error de conexión o no se detectó Arduino: {e}.")
    print("El script intentará cargar 'datos_torricelli.csv' de una ejecución anterior...\n")

# --- 4. GUARDADO DE DATOS CRUDOS Y ANÁLISIS ---
directorio_actual = os.getcwd()
archivo_csv = os.path.join(directorio_actual, "datos_torricelli.csv")

# Si se capturaron datos nuevos, se guardan y se usan
if len(tiempos) > 10:
    df = pd.DataFrame({'time_s': tiempos, 'distance_cm': distancias})
    df.to_csv(archivo_csv, index=False)
    print(f"\n[+] Datos crudos guardados exitosamente en: {archivo_csv}")
# Si no, intentar cargar el archivo CSV existente
elif os.path.exists(archivo_csv):
    print(f"Cargando datos existentes desde: {archivo_csv}")
    df = pd.read_csv(archivo_csv)
    tiempos = df['time_s'].tolist()
    distancias = df['distance_cm'].tolist()
else:
    print("No hay datos nuevos ni un archivo CSV previo. Saliendo...")
    df = pd.DataFrame()

# Proceder al ajuste y graficado si hay datos
if not df.empty:
    print("\nProcesando el ajuste no lineal...")
    t_arr = np.array(tiempos)
    d_arr = np.array(distancias)
    
    # 1. RECORTES DE TRANSITORIOS (Data Trimming)
    mask = (t_arr > 20.0) & (t_arr < 130.0)
    
    t_clean = t_arr[mask]
    d_clean = d_arr[mask]
    u_clean = np.full(len(d_clean), SIGMA_US)
    
    if len(t_clean) > 10:
        try:
            # Ajustamos solo la parte limpia
            popt, pcov = curve_fit(modelo_torricelli, t_clean, d_clean, 
                                   p0=[0.001, 0.01, d_inicial_promedio], 
                                   sigma=u_clean, absolute_sigma=True)
            perr = np.sqrt(np.diag(pcov))
            
            d_fit_completo = modelo_torricelli(t_arr, *popt)
            d_fit_clean = modelo_torricelli(t_clean, *popt)
            
            diferencias = d_arr - d_fit_completo
            residuos_normalizados = diferencias / SIGMA_US
            
            chi2 = np.sum(((d_clean - d_fit_clean) / u_clean)**2)
            ndof = len(t_clean) - 3  
            chi2_ndof = chi2 / ndof if ndof > 0 else 0

            print("\n--- EXTRACCIÓN PARAMÉTRICA ---")
            print(f"Distancia inicial (C): {popt[2]:.2f} ± {perr[2]:.2f} cm")
            print(f"Término lineal (B):    {popt[1]:.4f} ± {perr[1]:.4f}")
            print(f"Término cuadrático (A):{popt[0]:.4f} ± {perr[0]:.4f}")
            print(f"Chi2/ndof:             {chi2_ndof:.2f}")

            # 2. GRÁFICA DE TRES PANELES
            fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 11), sharex=True)
            
            # Panel 1: Cinemática Macroscópica
            ax1.errorbar(t_arr, d_arr, yerr=SIGMA_US, fmt='.', color='gray', ecolor='gray', 
                         alpha=0.5, capsize=1, label='Datos Ultrasegundo (2$\sigma$)')
            ax1.scatter(t_arr, d_arr, color='c', s=15, alpha=0.3)
            t_fit_line = np.linspace(t_arr.min(), t_arr.max(), 500)
            ax1.plot(t_fit_line, modelo_torricelli(t_fit_line, *popt), color='red', linewidth=2.5, 
                     label=f'Ajuste Torricelli\n$\chi^2/ndof = {chi2_ndof:.2f}$')
            ax1.set_title("Cinemática de Vaciado de Tanque (Ley de Torricelli)")
            ax1.set_ylabel("Distancia Sensor-Agua $d_{real}$ (cm)")
            ax1.legend(loc='upper left')
            ax1.set_ylim(8.3, 15.8)
            
            # Panel 2: Diferencia absoluta
            ax2.errorbar(t_arr, diferencias, yerr=SIGMA_US, fmt='.', color='gray', ecolor='gray', 
                         alpha=0.5, capsize=1)
            ax2.scatter(t_arr, diferencias, color='c', s=15, alpha=0.3)
            ax2.axhline(0, color='black', linewidth=1.5)
            ax2.set_ylabel("Diferencia\n$d_{real} - d_{fit}$ (cm)")
            ax2.set_ylim(-1.3, 1.3)
            
            # Panel 3: Residuos normalizados
            ax3.errorbar(t_arr, residuos_normalizados, yerr=1.0, fmt='.', color='gray', ecolor='gray', 
                         alpha=0.5, capsize=1)
            ax3.scatter(t_arr, residuos_normalizados, color='c', s=15, alpha=0.3)
            ax3.axhline(0, color='black', linewidth=1.5)
            ax3.axhline(2, color='red', linestyle='--', alpha=0.6)
            ax3.axhline(-2, color='red', linestyle='--', alpha=0.6)
            ax3.set_ylabel("Residuos\nnormalizados $r_i$")
            ax3.set_xlabel("Tiempo $t$ (s)")
            ax3.set_ylim(-4.5, 5.5)
            
            plt.xlim(0, 130)
            plt.tight_layout()
            plt.show()
            
        except Exception as e:
            print(f"Fallo en el ajuste de curva: {e}")
    else:
        print("El recorte dejó muy pocos datos para ajustar. Verifica los tiempos de la máscara.")